In [65]:
import random
import copy

In [72]:
COLOURS = ['Red', 'Blue', 'Green', 'Yellow']
NUMBERS = list(range(10))         
PLAYERS = ['p1', 'p2', 'p3']  
PLAYER_NAMES = {'p1': 'P1 (Minimax Defensive)',
    'p2': 'P2 (Expectimax Offensive)' ,
    'p3': 'P3',
}

DEFAULT_SEED = 42

In [73]:
class Card:# single uno card

    def __init__(self, colour, value):
        self.colour = colour
        self.value = value           

        
    def __repr__(self):
        return f"{self.colour} {self.value}"

    def __eq__(self, other):
        return (isinstance(other, Card)and self.colour == other.colour and self.value == other.value)

    def __hash__(self):
        return hash((self.colour, self.value))

    def __deepcopy__(self, memo):
        #colour value dono fixed tu no need to recurse
        return Card(self.colour, self.value)



    def matches(self, top: 'Card'): #same colour or number
        return self.colour == top.colour or self.value == top.value
        
    def is_skip(self):
        return self.value == 'Skip'

#DEBUG
c1 = Card('Red' , 8)
print(c1.is_skip())

c2 = Card('Yellow ', 'Skip')
print(c2.is_skip())

c1.__repr__()

False
True


'Red 8'

In [74]:
def deck_generator(): #total 4×11 so 44cards 0 say 9 sab mai and 1 skip bhi so 11 

    deck = []
    for colour in COLOURS:
        for num in NUMBERS:                  
            deck.append(Card(colour, num))
        deck.append(Card(colour, 'Skip'))   

        
    random.shuffle(deck)
    return deck



#DEBUG
#deck= deck_generator()
#print(deck)
#len(deck)

In [75]:
def get_valid_moves(hand, top_card):
    return [card for card in hand if card.matches(top_card)]

#DEBUG
hand = [Card('Red', 6) , Card('Blue', 3), Card('Yellow', 4) , Card('Yellow', 'Skip') , Card('Green', 'Skip') ]
top = Card('Yellow', 6)
get_valid_moves(hand , top)

[Red 6, Yellow 4, Yellow Skip]

In [76]:
def apply_move(state, move):#player key is Player Names ki keys p1 ,p2 , p3

    player_key, card = move
    
    state_updated = copy.deepcopy(state)#deep copy takay original wala is not mutated
    skip_player= None

    if card is None:#card nai hai valid also reshuffle already played ya discards into deck agar deck hi khali hogaya
        if not state_updated['deck'] and state_updated.get('discards'):
            state_updated['deck'] = state_updated['discards']
            random.shuffle(state_updated['deck'])
            state_updated['discards'] = []

        
        if state_updated['deck']:
            drawn = state_updated['deck'].pop()
            state_updated[player_key].append(drawn)
       

    else:
        hand = state_updated[player_key]
        for i, c in enumerate(hand):
            if c == card:
                hand.pop(i)
                break

        if 'discards' not in state_updated:#purana top ko discard mai 
            state_updated['discards'] = []
            
        state_updated['discards'].append(state_updated['top_card'])

        state_updated['top_card'] = card


        
        if card.is_skip():
            index = PLAYERS.index(player_key)
            skip_player = PLAYERS[(index + 1) % len(PLAYERS)]


    
    return state_updated, skip_player

In [82]:
def evaluate(state, player_key, strategy= 'defensive'):
    opp_keys = [k for k in PLAYERS if k != player_key]
    cai  = len(state[player_key])
    copp = sum(len(state[k]) for k in opp_keys) / 2.0
    s    = sum(1 for c in state[player_key] if c.is_skip())

    if strategy == 'defensive':#penalise own hand harder and reward skips highly
        return 50.0 - 6.0 * cai + 2.0 * copp + 4.0 * s
    else:#moderate self-penalty adn strongly reward opponent burden
        return 50.0 - 5.0 * cai + 3.0 * copp + 2.0 * s

In [87]:
class TreeNode:
    
    def __init__(self, label, node_type, score = None):
        self.label     = label
        self.node_type = node_type   # max , min ,chance,opp,leaf, terminal pruned
        self.children = []
        self.score     = score
        


    
    def add_child(self, child: 'TreeNode'):
        self.children.append(child)




def print_tree(node, prefix = "",is_root= True, is_last= True, max_depth = 3,cur_depth= 0):#recursive
   
    if node.score is not None:
        score_str = f" [{node.score:+.1f}]"
    else:
        score_str = " "
    if is_root:
        print(f"[{node.node_type:8s}] {node.label}{score_str}")
        child_prefix = ""
    else:
        if is_last:
            conn = "|__ "
        else:
            conn = "|-- "
        print(f"{prefix}{conn}[{node.node_type:8s}] {node.label}{score_str}")
        if is_last:
            child_prefix = prefix + "    "
        else:
            child_prefix = prefix + "|   "


        
    if cur_depth >= max_depth:
        if node.children:
            print(f"{child_prefix}... ({len(node.children)} children not shown)")
        return



    
    for i, child in enumerate(node.children):
        print_tree(child, child_prefix, is_root=False,is_last=(i == len(node.children) - 1), max_depth=max_depth, cur_depth=cur_depth + 1)
print("complete process of ai decisions above")



complete process of ai decisions above


**EVALUATION FUNCTION EXPLANATION**

Base formula (from assignment) Score = 50 − 5·(CAI) + 2·(Copp) + 3·(S)
    CAI  = number of cards in the AI player's hand  (lower -> better)
    Copp = average cards held by the two opponents   (higher -> better)
    S    = number of Skip cards in the AI's hand     (higher -> better)

The formula tuned differently per strategy
DEFENSIVE (Player 1 Minimax)
  • owncard penalty raised  (−6 instead of −5) defensive play is obsessed with keeping the hand small
  • skip reward raised (+4 instead of +3) skip cards are gold they freeze opponents at critical moments
  • opponent burden kept moderate (+2) we care less about loading opponents and more about protecting ourself
  Formula -> 50 − 6·CAI + 2·Copp + 4·S



OFFENSIVE (Player 2 Expectimax)
  • owncard penalty standard (−5)we still want to shed cards, but not at the expense of loading opponents
  • Opponent burden raised (+3 instead of +2)offensive play tries to force opponents to have MORE cards (draw situations)
  • Skip reward lowered (+2)skips are useful but we prefer raw card-shedding over stalling
  Formula -> 50 − 5·CAI + 3·Copp + 2·S

In [89]:
def minimax(state,depth, player_key, current_player,skipped, alpha = float('-inf'), beta = float('inf'), build_tree = True):
 
    for p in PLAYERS:
        if len(state[p]) == 0:
            s = (1000.0 + depth) if p == player_key else (-1000.0 - depth)  
            node = TreeNode(f"WIN({p.upper()})", "TERMINAL", s) if build_tree else None
            return s, None, node

    if depth == 0:
        s = evaluate(state, player_key, 'defensive')
        node = TreeNode("Leaf", "LEAF", s) if build_tree else None

        
        return s, None, node

    #skip propagation
    if current_player in skipped:
        new_skipped = skipped - {current_player}
        index = PLAYERS.index(current_player)
        next_index = PLAYERS[(index + 1) % len(PLAYERS)]
        return minimax(state, depth, player_key, next_index, new_skipped,alpha, beta, build_tree)

    valid   = get_valid_moves(state[current_player], state['top_card'])
    if valid:
        actions = valid
    else:
        actions = [None] 

    is_max    = (current_player == player_key)
    if is_max:
        node_type = "MAX"
    else:
        node_type = "MIN"
    
    index = PLAYERS.index(current_player)
    next_index = PLAYERS[(index + 1) % len(PLAYERS)]

    root_node = (TreeNode(f"{current_player.upper()} | top={state['top_card']}",  node_type) if build_tree else None)

    if is_max:
        best_score = float('-inf')
    else:
        best_score = float('inf')
        
    if actions:
        best_move = actions[0]
    else:
        best_move = None

    for action in actions:
        new_state, skip_p = apply_move(state, current_player, action)
        if skip_p:
            new_skipped = {skip_p}
        else:
            new_skipped = set()

        child_score, _, child_node = minimax(new_state, depth - 1, player_key, next_index, new_skipped,alpha, beta, build_tree)

        if build_tree:
            if action is not None:
                action_str = str(action)
            else:
                action_str = "Draw (forced)"
            anode = TreeNode(action_str, "ACTION", child_score)
            if child_node:
                anode.add_child(child_node) 
            root_node.add_child(anode)

        if is_max:
            if child_score > best_score:
                best_score = child_score
                best_move  = action
            alpha = max(alpha, best_score)
            
        else:
            if child_score < best_score:
                best_score = child_score
                best_move  = action
                
            beta = min(beta, best_score)

        
        # alphabeta pruning
        if beta <= alpha:
            if build_tree:
                root_node.add_child(TreeNode("(pruned)", "PRUNED"))
            break



    if build_tree:
        root_node.score = best_score
    return best_score, best_move, root_node